In [1]:
import pandas as pd
import numpy as np

ruta=r"C:\Users\josperez\Documents\A-DataStack\01-Proyectos\01-Data_PipelinesFibex\02_Data_Lake\gold_data\SLA_Gold.parquet"
sla = pd.read_parquet(ruta)

# Asegurar que la columna de fecha sea datetime (evita comparaciones con strings)
sla["Fecha Finalizacion"] = pd.to_datetime(sla["Fecha Finalizacion"], errors="coerce")

decem = sla[sla["Fecha Finalizacion"] >= pd.Timestamp('2026-01-01')]

# Calcular medias solo para columnas numéricas y de tipo timedeltas (SLA)
num_mean = decem.select_dtypes(include=[np.number]).mean()
td_mean = decem.select_dtypes(include=['timedelta64[ns]']).mean()
mean_sla = pd.concat([num_mean, td_mean])

print("Mean SLA:")
print(mean_sla)

KeyError: 'Fecha Finalizacion'

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Definir tu Top 5 basado en el análisis de Pareto
top_5_soluciones = [
    'CONECTOR DAÑADO',
    'CAMBIO DE VLAN',
    'CONFIGURACION DE EQUIPO',
    'FALLA GENERAL',
    'FALLA LOS'
]

# 2. Filtrar el DataFrame limpio (df_final) para quedarnos solo con el Top 5
df_pareto = df_final[df_final['Solucion Aplicada'].isin(top_5_soluciones)].copy()

# 3. Graficar los histogramas separados para ver la "limpieza" de las curvas
g = sns.FacetGrid(df_pareto, col="Solucion Aplicada", col_wrap=3, height=4, sharex=False, sharey=False)
g.map_dataframe(sns.histplot, x="SLA Resolucion Min", kde=True, bins=30, color='teal')

# Dar formato a los gráficos
g.set_titles(col_template="{col_name}")
g.set_axis_labels("Minutos de Resolución", "Cantidad de Tickets")
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np

# Supongamos que df_pareto es tu tabla ya filtrada con el Top 5 de soluciones
# 1. Definir el tamaño del "depósito" o bin (ej. cada 60 minutos = 1 hora)
ancho_bin = 60
max_minutos = int(df_pareto['SLA Resolucion Min'].max())
bins = range(0, max_minutos + ancho_bin, ancho_bin)

# 2. Crear las etiquetas (ej. "0-59", "60-119", etc.)
etiquetas = [f"{i} a {i+ancho_bin-1}" for i in bins[:-1]]

# 3. Asignar cada ticket a su rango correspondiente
df_pareto['Rango_Tiempo_Min'] = pd.cut(df_pareto['SLA Resolucion Min'], bins=bins, labels=etiquetas, right=False)

# 4. AGRUPAR: Esta es la magia que comprime los datos para Power BI
df_gold_distribucion = df_pareto.groupby(['Solucion Aplicada', 'Rango_Tiempo_Min']).size().reset_index(name='Cantidad_Tickets')

# Limpiar rangos donde no hubo ningún ticket para ahorrar aún más peso
df_gold_distribucion = df_gold_distribucion[df_gold_distribucion['Cantidad_Tickets'] > 0]
display(df_gold_distribucion)
# Exportar a Parquet
# df_gold_distribucion.to_parquet("Gold_Distribucion_SLA.parquet")

In [ ]:
import pandas as pd
df = pd.read_parquet(r'C:\Users\josperez\Documents\A-DataStack\01-Proyectos\01-Data_PipelinesFibex\02_Data_Lake\gold_data\Ventas_Listado_Gold.parquet')
display(df)

In [ ]:
import pandas as pd
df = pd.read_parquet(r'C:\Users\josperez\Documents\A-DataStack\01-Proyectos\01-Data_PipelinesFibex\02_Data_Lake\gold_data\Recaudacion_Gold.parquet')
print(df.columns)
df = df.groupby(['Hora de Pago', 'Vendedor'], as_index=False).size()


display(df)

In [ ]:
import polars as pl
import pandas as pd
df = pd.read_parquet(r'C:\Users\josperez\Downloads\Ventas_Estatus_Gold-Sharepoint.parquet')

mask = (df['Fecha']>=('2026-02-01')) & (df['Fecha']<=('2026-02-28')) &(df['Vendedor'].str.contains("OFI", case=False, na=False))
df_feb=df[mask]
print(df_feb.columns)
df_feb = df_feb.drop_duplicates(subset=['N° Abonado', 'Documento','Fecha','Vendedor','Ciudad','Hora'])
display(df_feb.describe(include='all'))
display(df_feb)

In [ ]:
import pandas as pd
import polars as pl
from pathlib import Path
import duckdb

rutahome = Path.home()
ruta_base = rutahome / 'Documents' / 'A-DataStack' / '01-Proyectos' / '01-Data_PipelinesFibex' / '02_Data_Lake' / 'gold_data'

lf = duckdb.query


In [5]:
import pandas as pd 

df_ventas= pd.read_parquet(r"C:\Users\josperez\Documents\A-DataStack\01-Proyectos\01-Data_PipelinesFibex\02_Data_Lake\gold_data\Ventas_Listado_Gold.parquet")
df_estatus= pd.read_parquet(r"C:\Users\josperez\Documents\A-DataStack\01-Proyectos\01-Data_PipelinesFibex\02_Data_Lake\gold_data\Ventas_Estatus_Gold.parquet")
df_afluencia = pd.read_parquet(r"C:\Users\josperez\Documents\A-DataStack\01-Proyectos\01-Data_PipelinesFibex\02_Data_Lake\gold_data\Afluencia_Gold.parquet")
#display(df_estatus)
display(df_afluencia.info())
fecha = pd.to_datetime('2026-04-01')
mask_ofi_march = (df_ventas['Canal'] == 'OFICINA COMERCIAL') & (df_ventas['Fecha Contrato'] >= fecha)
mask_est_march = (df_estatus['Fecha'] >= fecha)
mask_ventas_march = (df_afluencia['Fecha']>=fecha) & (df_afluencia['Tipo de afluencia'] == 'Ventas')

df_ventas = df_ventas[mask_ofi_march]
df_estatus= df_estatus[mask_est_march]
df_afluencia = df_afluencia[mask_ventas_march]



display(df_ventas.groupby(['Estatus']).agg(Cantidad = ('Estatus','size')))
display(df_estatus.groupby(['Estatus']).agg(Cantidad = ('Estatus','size')))
display(df_afluencia.groupby(['Estatus']).agg(Cantidad = ('Estatus','size')))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1680636 entries, 0 to 1680635
Data columns (total 15 columns):
 #   Column             Non-Null Count    Dtype                 
---  ------             --------------    -----                 
 0   N° Abonado         1680636 non-null  string                
 1   Documento          21112 non-null    string                
 2   Estatus            1680636 non-null  string                
 3   Fecha              1680636 non-null  timestamp[ns][pyarrow]
 4   Vendedor           1680636 non-null  string                
 5   Grupo Afinidad     1680636 non-null  string                
 6   Nombre Franquicia  1680636 non-null  string                
 7   Ciudad             1680636 non-null  string                
 8   Tipo de afluencia  1680636 non-null  string                
 9   Oficina            1680623 non-null  object                
 10  Clasificacion      1659524 non-null  string                
 11  Hora               21112 non-null    

None

,Cantidad
Estatus,
ACTIVO,344
OBSTRUCCION,5
POR IMPLEMENTACION,16
POR INSTALAR,114
POR VGT,1
SIN FACTIBILIDAD,5


,Cantidad
Estatus,
ACTIVO,349
ANULADO,6
OBSTRUCCION,5
POR IMPLEMENTACION,16
POR INSTALAR,106
POR VGT,1
RECHAZADO,2
REEMBOLSO ADM,3
REEMBOLSO PAGADOS,1


,Cantidad
Estatus,
ACTIVO,349
ANULADO,6
OBSTRUCCION,5
POR IMPLEMENTACION,16
POR INSTALAR,106
POR VGT,1
RECHAZADO,2
REEMBOLSO ADM,3
REEMBOLSO PAGADOS,1


In [1]:
import pandas as pd


df = pd.read_parquet(r'C:\Users\josperez\Documents\A-DataStack\01-Proyectos\01-Data_PipelinesFibex\02_Data_Lake\bronze_data\Horas_Raw_Bronze.parquet')
df_recaudacion = pd.read_parquet(r'C:\Users\josperez\Documents\A-DataStack\01-Proyectos\01-Data_PipelinesFibex\02_Data_Lake\bronze_data\Recaudacion_Raw_Bronze.parquet')
display(df.columns)
display(df)
display(df_recaudacion.columns)
display(df_recaudacion)
df_merge = pd.merge(df_recaudacion, df, on='ID Pago', how='inner')
display(df_merge.head())

Index(['ID Contrato', 'ID Pago', 'N° Abonado', 'Documento', 'Nro Recibo',
       'Fecha', 'Hora de Pago', 'Oficina Cobro', 'Suscripción', 'Source.Name',
       'Fecha_Modificacion_Archivo', 'ID pago', 'Cobrador', 'Forma de Pago',
       'Cliente', 'Total Pago', 'Total Pago Bs', 'Tasa de cambio del día',
       'Usuario de Anulacion'],
      dtype='object')

,ID Contrato,ID Pago,N° Abonado,Documento,Nro Recibo,Fecha,Hora de Pago,Oficina Cobro,Suscripción,Source.Name,Fecha_Modificacion_Archivo,ID pago,Cobrador,Forma de Pago,Cliente,Total Pago,Total Pago Bs,Tasa de cambio del día,Usuario de Anulacion
0,'CONT772929D6D7C85860','6977B69642C0B8654827',1253,8366290,014964,2026-01-26 00:00:00,1899-12-31 14:48:00,OFC - FIBEX ARAGUA-MATURIN,30,Data-HorasPago 30-1 al 2-2.xlsx,2026-02-02 08:53:00.515741,None,None,None,None,None,None,None,None
1,'CONTD365420046F15117','69971B201915B3028843',LCH49718,11382001,525353,2026-02-19 00:00:00,1899-12-31 10:17:00,OFIC-SAN DIEGO-EL RINCON,30,Data-HorasPago 19-2 al 19-2.xlsx,2026-02-19 16:23:10.719427,None,None,None,None,None,None,None,None
2,'CONT1239DC230BC13727','CONT1239DC230BC13727',BQ2870,17229348,183692,2026-02-09 00:00:00,1899-12-31 16:36:00,OFIC-METROPOLIS-BQTO,25,Data-HorasPago 9-2 al 10-2.xlsx,2026-02-10 08:35:12.669482,None,None,None,None,None,None,None,None
3,'CONT690C1B08CE070480','69690CD6B76240852888',VC6426,19060999,063551,2026-01-15 00:00:00,1899-12-31 11:52:00,OFICINA VILLA DE CURA,23.2,Data-HorasPago 30-1 al 2-2.xlsx,2026-02-02 08:53:00.515741,None,None,None,None,None,None,None,None
4,'CONT35B1140825960967','69DCC513C806A0190415',LCH4212,27652247,910880,13/04/2026,06:27 AM,OFI-VIRTUAL,32.5,Data - HorasPago 13-04-2026 al 14-04-2026.xlsx,2026-04-14 16:08:01.185838,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3020800,'CONT06508B1B30A63471','69C15DA3288057485882',LCH55127,10957321,288019,2026-03-23 00:00:00,1899-12-31 11:38:00,OFI-PTO LA CRUZ,30,Data-HorasPago 23-3 al 24-3.xlsx,2026-03-24 10:58:07.044243,None,None,None,None,None,None,None,None
3020801,'CONTD2E8C268F4786717','68922509540525333975',C0971,None,None,2025-08-05 00:00:00,1899-12-31 11:39:00,OFIC GALERIA EL PARAISO,None,Data-HorasPago 01-08 al 27-11.xlsx,2025-12-02 10:03:02.810474,68922509540525301760,SUJAILETH MAIROBIS GARCIA OFIC GALERIAS PARAISO,EFECTIVO DOLLAR - 50.00,None,None,None,None,None
3020802,'CONT6F8975BC1F406007','696F9909D0AA77252940',FC2477,27428501,010713,2026-01-20 00:00:00,1899-12-31 11:02:00,OFC-CUMANACOA,None,Data-HorasPago 20-1 al 20-1.xlsx,2026-01-20 16:53:28.533969,None,None,None,None,None,None,None,None
3020803,'CONT40DDF2975CF01021','6989E7938EE7E6416359',LAG0127,21135779,012861,2026-02-09 00:00:00,1899-12-31 10:00:00,OFC - LAGUNITAS,30,Data-HorasPago 9-2 al 10-2.xlsx,2026-02-10 08:35:12.669482,None,None,None,None,None,None,None,None


Index(['ID Contrato', 'ID Pago', 'N° Abonado', 'Nro Recibo', 'Fecha',
       'Total Pago', 'Total Pago Bs', 'Estatus Pago', 'Forma de Pago',
       'Monto Forma Pago', 'Tasa de cambio del día', 'Banco', 'Referencia',
       'Cobrador', 'Nombre Caja', 'Oficina Cobro', 'Fecha Contrato', 'Estatus',
       'Suscripción', 'Etiqueta', 'Grupo Afinidad', 'Nombre Franquicia',
       'Ciudad', 'Source.Name', 'Fecha_Modificacion_Archivo', 'ID pago'],
      dtype='object')

,ID Contrato,ID Pago,N° Abonado,Nro Recibo,Fecha,Total Pago,Total Pago Bs,Estatus Pago,Forma de Pago,Monto Forma Pago,...,Fecha Contrato,Estatus,Suscripción,Etiqueta,Grupo Afinidad,Nombre Franquicia,Ciudad,Source.Name,Fecha_Modificacion_Archivo,ID pago
0,'CONTF91462D14FB42393','6906207AE6E301052143',V23130,199828,2025-11-01,30,6718.87,PAGADO,TARJETA DE DEBITO,30,...,2023-02-24 00:00:00,ACTIVO,30,None,FIBEX EXPRESS,FIBEX VALENCIA,VALENCIA,Data - Recaudacion 1-11-2025 al 15-11-2025.xlsx,2026-01-14 13:31:59.940364,6906207AE6E301052143
1,'CONTD87627486E485529','69061B8D3D8572666742',GU34478,325426,2025-11-01,10,2240,PAGADO,TARJETA DE DEBITO,10,...,2025-03-17 00:00:00,ACTIVO,20,KARLA REBOLLEDO,FIBEX EXPRESS,FIBEX GUARICO,ELSOMBRERO,Data - Recaudacion 1-11-2025 al 15-11-2025.xlsx,2026-01-14 13:31:59.940364,69061B8D3D8572666742
2,'CONT5855071FF1E85365','690627C515CC77490487',LCH25171,434319,2025-11-01,4.92,1101.89,PAGADO,TARJETA DE DEBITO,4.92,...,2024-05-30 00:00:00,CORTADO,30,None,FIBEX EXPRESS,FIBEX ANZOATEGUI,CUMANA,Data - Recaudacion 1-11-2025 al 15-11-2025.xlsx,2026-01-14 13:31:59.940364,690627C515CC77490487
3,'CONT923CFFCDE3914131','690602BEF13EF0368020',V6820,199589,2025-11-01,20,4479.24,PAGADO,TARJETA DE CREDITO,20,...,2022-05-28 00:00:00,ACTIVO,20,None,FIBEX EXPRESS,FIBEX VALENCIA,VALENCIA,Data - Recaudacion 1-11-2025 al 15-11-2025.xlsx,2026-01-14 13:31:59.940364,690602BEF13EF0368020
4,'CONT6C0A054B90092366','69061987EAF732248447',V4931,199768,2025-11-01,20.88,4676.45,PAGADO,TARJETA DE DEBITO,20.88,...,2022-04-29 00:00:00,ACTIVO,20,None,FIBEX EXPRESS,FIBEX VALENCIA,VALENCIA,Data - Recaudacion 1-11-2025 al 15-11-2025.xlsx,2026-01-14 13:31:59.940364,69061987EAF732248447
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2327726,'CONT9E185B34F1A89935','69B009F17B9315998931',MGTA9736,095975,2026-03-10,25,10499.68,PAGADO,TARJETA DE DEBITO,25,...,2025-02-01 00:00:00,ACTIVO,25,YUSRODRIGUEZ,HOGAR,FIBEX MARGARITA,MARGARITA,Data - Recaudacion 9-3-2026 al 10-3-2026.xlsx,2026-03-10 08:12:53.422155,69B009F17B9315998931
2327727,'CONT1CABC4DCD3688860','69B00A13286B11905410',GU53025,423460,2026-03-10,29.41,12352.89,PAGADO,TARJETA DE DEBITO,29.41,...,2025-11-18 00:00:00,ACTIVO,30,DIANNY MARCANO,HOGAR,FIBEX GUARICO,VALLE DE LA PASCUA,Data - Recaudacion 9-3-2026 al 10-3-2026.xlsx,2026-03-10 08:12:53.422155,69B00A13286B11905410
2327728,'CONT23F867EA84623312','69B00A7F4CAAD1920452',LCH36893,537478,2026-03-10,35,14700,PAGADO,TARJETA DE DEBITO,35,...,2024-10-31 00:00:00,ACTIVO,35,None,HOGAR,FIBEX ANZOATEGUI,CUMANA,Data - Recaudacion 9-3-2026 al 10-3-2026.xlsx,2026-03-10 08:12:53.422155,69B00A7F4CAAD1920452
2327729,'CONTC822D1A0D3041096','69B00A0C7CD036738833',GU50582,423451,2026-03-10,10,4200,PAGADO,TARJETA DE DEBITO,10,...,2025-09-15 00:00:00,ACTIVO,25,DAISY RODRíGUEZ,HOGAR,FIBEX GUARICO,TUCUPIDO,Data - Recaudacion 9-3-2026 al 10-3-2026.xlsx,2026-03-10 08:12:53.422155,69B00A0C7CD036738833


,ID Contrato_x,ID Pago,N° Abonado_x,Nro Recibo_x,Fecha_x,Total Pago_x,Total Pago Bs_x,Estatus Pago,Forma de Pago_x,Monto Forma Pago,...,Source.Name_y,Fecha_Modificacion_Archivo_y,ID pago_y,Cobrador_y,Forma de Pago_y,Cliente,Total Pago_y,Total Pago Bs_y,Tasa de cambio del día_y,Usuario de Anulacion
0,'CONTF91462D14FB42393','6906207AE6E301052143',V23130,199828,2025-11-01,30,6718.87,PAGADO,TARJETA DE DEBITO,30,...,Data-HorasPago 01-08 al 27-11.xlsx,2025-12-02 10:03:02.810474,6906207AE6E301052143,GLEYSI BENAVIDE OFIC TORRE FIBEX VIñEDO,TARJETA DE DEBITO - 30.00,None,None,None,None,None
1,'CONTD87627486E485529','69061B8D3D8572666742',GU34478,325426,2025-11-01,10,2240,PAGADO,TARJETA DE DEBITO,10,...,Data-HorasPago 01-08 al 27-11.xlsx,2025-12-02 10:03:02.810474,69061B8D3D8572666742,AGENTE AUTORIZADO FIBERWAVE YOXIMAR YOXENIA LO...,TARJETA DE DEBITO - 10.00,None,None,None,None,None
2,'CONT5855071FF1E85365','690627C515CC77490487',LCH25171,434319,2025-11-01,4.92,1101.89,PAGADO,TARJETA DE DEBITO,4.92,...,Data-HorasPago 01-08 al 27-11.xlsx,2025-12-02 10:03:02.810474,690627C515CC77490487,AURIMIR DEL VALLE GUEVARA OFIC CUMANA,TARJETA DE DEBITO - 4.92,None,None,None,None,None
3,'CONT923CFFCDE3914131','690602BEF13EF0368020',V6820,199589,2025-11-01,20,4479.24,PAGADO,TARJETA DE CREDITO,20,...,Data-HorasPago 01-08 al 27-11.xlsx,2025-12-02 10:03:02.810474,690602BEF13EF0368020,JHORIANNY ESPINOZA OFIC TORRE FIBEX VIñEDO,TARJETA DE CREDITO - 20.00,None,None,None,None,None
4,'CONT6C0A054B90092366','69061987EAF732248447',V4931,199768,2025-11-01,20.88,4676.45,PAGADO,TARJETA DE DEBITO,20.88,...,Data-HorasPago 01-08 al 27-11.xlsx,2025-12-02 10:03:02.810474,69061987EAF732248447,JHORIANNY ESPINOZA OFIC TORRE FIBEX VIñEDO,TARJETA DE DEBITO - 20.88,None,None,None,None,None


In [2]:
import pandas as pd
df = pd.read_parquet(r'C:\Users\josperez\Documents\A-DataStack\01-Proyectos\01-Data_PipelinesFibex\02_Data_Lake\gold_data\Recaudacion_Gold.parquet')
display(df)

,ID Contrato,N° Abonado,Fecha,Total Pago,Forma de Pago,Banco,Oficina,Fecha Contrato,Estatus,Suscripción,Grupo Afinidad,Nombre Franquicia,Ciudad,Vendedor,Tipo de afluencia,Clasificacion,Hora de Pago
0,'CONT21FE5D5030705988',LCH58251,2025-07-01,16.64,TARJETA DE DEBITO,PROVINCIAL,OFC COMERCIAL CUMANA,2025-05-12 00:00:00,ACTIVO,30,HOGAR,FIBEX ANZOATEGUI,CUMANA,JORGE PAREJO OFIC CUMANA,RECAUDACIÓN,OFICINAS PROPIAS,07:00
1,'CONT82B7255301054845',GU17461,2025-07-01,30.00,TARJETA DE DEBITO,100% BANCO TH,OFC - VALLE LA PASCUA,2024-07-01 00:00:00,ACTIVO,30,HOGAR,FIBEX GUARICO,VALLE DE LA PASCUA,GERONIMO A ZAMBRANO ASESOR OFICINA GUARICO,RECAUDACIÓN,ALIADOS Y DESARROLLO,08:00
2,'CONT3CA2F6D7A1834761',PTO10344,2025-07-01,30.00,TARJETA DE DEBITO,PROVINCIAL,OFI-PTO CABELLO,2024-11-19 00:00:00,ACTIVO,30,HOGAR,FIBEX PTO CABELLO,PUERTO CABELLO,CARILYN MELENDEZ OFIC PUERTO CABELLO,RECAUDACIÓN,OFICINAS PROPIAS,08:00
3,'CONT9E17A54A36F85147',LV2565,2025-07-01,20.00,TARJETA DE DEBITO,BANCO NACIONAL CREDITO,OFC-INTERCOM SAN CARLOS LAS VEGAS,2025-02-01 00:00:00,ACTIVO,25,HOGAR,FIBEX LAS VEGAS,LAS VEGAS,INES MARIA LABARCA BELTRAN AGENTE AUTORIZADO I...,RECAUDACIÓN,ALIADOS Y DESARROLLO,08:00
4,'CONT0F889188DCA55438',LV3444,2025-07-01,10.00,TARJETA DE DEBITO,BANCO NACIONAL CREDITO,OFC-INTERCOM SAN CARLOS LAS VEGAS,2025-04-28 00:00:00,CORTADO,25,HOGAR,FIBEX LAS VEGAS,LAS VEGAS,INES MARIA LABARCA BELTRAN AGENTE AUTORIZADO I...,RECAUDACIÓN,ALIADOS Y DESARROLLO,08:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1537689,'CONT6625705F0D043154',FM1739,2026-03-06,35.00,TARJETA DE DEBITO,PROVINCIAL,OFC COBRO CENTRO,2023-11-28 00:00:00,ACTIVO,35,HOGAR,FIBEX MATURIN,MATURIN,ELOINA ARTEAGA,RECAUDACIÓN,ALIADOS Y DESARROLLO,13:00
1537690,'CONTC07586C5D9929991',V9476,2026-03-06,39.99,TARJETA DE DEBITO,100% BANCO,OFI-METROPOLIS,2022-07-02 00:00:00,ACTIVO,42,HOGAR,FIBEX VALENCIA,VALENCIA,MAYERLIN GARCIA WALTER OFIC METROPOLIS VALENCIA,RECAUDACIÓN,OFICINAS PROPIAS,13:00
1537691,'CONTB747062EC6A45817',C2933,2026-03-06,40.00,EFECTIVO DOLLAR,,OFI-PARAISO,2023-01-05 00:00:00,ACTIVO,42,HOGAR,FIBEX CARACAS,CARACAS,KARIANNYS RODRIGUEZ OFIC EL PARAISO CARACAS,RECAUDACIÓN,OFICINAS PROPIAS,13:00
1537692,'CONTA384F1D4C4149627',V122291,2026-03-06,25.00,TARJETA DE DEBITO,PROVINCIAL,OFC - GUACARA,2025-08-18 00:00:00,ACTIVO,25,HOGAR,FIBEX VALENCIA,VALENCIA,YELIUSKAR MELENDEZ AGENTE ESCALA 23 GUACARA,RECAUDACIÓN,ALIADOS Y DESARROLLO,13:00
